# SQL-Analyst Agent — Colab Training Notebook

Reproduces this repo's training runs end to end: clones the repo, installs dependencies, then walks through the SFT arm (`collect_sft_data.py` → `train_sft.py` → held-out eval) and the GRPO arm (`train_grpo.py` → `evaluate_grpo.py`).

**Before running anything:**
1. `Runtime > Change runtime type` → pick a GPU (T4 or better).
2. Add a Colab secret named exactly `WANDB_API_KEY` (key icon, left sidebar) with your Weights & Biases API key.

**One interruption:** after the install cells, you'll be asked to restart the runtime once. This is expected and normal. See the markdown cell where it happens for why. Everything before that point only needs to run once; after restarting, skip straight past it to the cell right after.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. Runtime > Change runtime type > pick a GPU, then re-run."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import os

REPO_URL = "https://github.com/AryamanJaggi/sql-agent-rlvr.git"

if not os.path.exists("sql-agent-rlvr"):
    !git clone {REPO_URL}
%cd sql-agent-rlvr
!git pull

In [ ]:
!pip install -q --retries 5 --timeout 120 -r requirements.txt
!pip install -q --retries 5 --timeout 120 unsloth vllm trl wandb

In [ ]:
# Unsloth checks the installed vLLM build against this runtime's CUDA
# version and blocks the import if they don't match.
!pip install -q --force-reinstall --no-deps https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl

## Restart runtime now

**Runtime → Restart session**

This is required because Unsloth's import-compatibility check patches Python's import machinery for the rest of the process once it runs, so simply having reinstalled the correct vLLM build above isn't enough within the same running kernel. The block has to be cleared by actually restarting.

**After restarting, continue from the next cell below.** Do not re-run the cells above (clone/install).

In [ ]:
%cd sql-agent-rlvr

import torch
from unsloth import FastLanguageModel
from vllm import SamplingParams
print("Imports OK - vLLM/CUDA mismatch is cleared.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = "/content/drive/MyDrive/sql_agent_rlvr_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved under:", RESULTS_DIR)

## SFT: collect_sft_data.py → train_sft.py → eval

Runs the untrained prompted baseline over Spider's `train` split (hard/extra difficulty tiers), keeps only its successful trajectories, fine-tunes a LoRA adapter on them, then evaluates that adapter on the `validation` split through the same ReAct harness the baseline was measured with.

Each step below runs a small test first to ensure there are no errors, then the real run.

In [ ]:
!python -m train.collect_sft_data --limit 3 --output {RESULTS_DIR}/sft_data_smoke.jsonl

In [ ]:
!python -m train.collect_sft_data --limit 150 --output {RESULTS_DIR}/sft_data.jsonl

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data_smoke.jsonl --output-dir {RESULTS_DIR}/sft_adapter_smoke --epochs 1

In [ ]:
!python -m train.train_sft --data {RESULTS_DIR}/sft_data.jsonl --output-dir {RESULTS_DIR}/sft_adapter

In [ ]:
!python -m eval.evaluate --policy unsloth --lora-path {RESULTS_DIR}/sft_adapter --split validation --limit 30 --wandb-project sql-agent-rlvr

## GRPO arm: train_grpo.py → evaluate_grpo.py

Trains via TRL's `environment_factory` (native tool-calling"), then evaluates with a matching native-tool-calling harness.

In [ ]:
!python -m train.train_grpo --limit 5 --num-generations 2 --epochs 1 --output-dir {RESULTS_DIR}/grpo_adapter_smoke

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter_smoke --split validation --limit 2 --max-steps 5

In [ ]:
!python -m train.train_grpo --limit 150 --output-dir {RESULTS_DIR}/grpo_adapter

In [ ]:
!python -m eval.evaluate_grpo --lora-path {RESULTS_DIR}/grpo_adapter --split validation --limit 30